## Auditing the inference: which tagging mechanisms earn their keep?

Three quarters of our Tesla sentences say "Tesla". The other quarter were tagged by **inference** --
a coreference model or a recency heuristic decided that "the company", "It" or "the automaker"
referred to Tesla. Every notebook so far has taken those tags on trust. Notebook 2.4 compared
*scorers* on a population the tagger had already defined; nothing has ever checked the tagger.

This notebook checks it, and the checking took three attempts. The first two were invalid because of
mistakes in the audit design -- both mine -- and both are documented here rather than quietly
discarded, because the way they failed is the most transferable thing in the notebook.

**Result up front:** coreference is good and stays, but the number below is only *part* of the
picture -- see the correction. The recency heuristic is worse than chance (13%) and has been
switched off; three mechanical substitution defects are fixed.

**⚠ CORRECTION (notebook 2.7, 2026-08-17).** The 93.5% figure below covers only the ~74% of
coref-tagged sentences that carry a usable span (see §5/§6's "honest caveat"). A follow-up audit of
the remaining no-span quarter, hand-labelled n=170 (up from an earlier n=50), found those rows
correct only **56.5% of the time** (95% CI [49.0%, 63.7%]) -- not a coin flip, but close, and well
short of 93.5%. The population-weighted **blended accuracy of the whole coref channel is 78.3%**,
not 93.5%. See `notebooks/text/2.7-aw-coref-verification.ipynb` for the full breakdown, error
taxonomy, and what changed as a result. Read this notebook's span-only numbers below as exactly
that -- span-only -- and go to 2.7 for the blended figure.

### 0. Setup

In [1]:
import numpy as np
import pandas as pd
from stock_predictor.config import DATA_DIR, PROJ_ROOT

pd.set_option("display.max_colwidth", 140)
sentences = pd.read_parquet(DATA_DIR / "sentences.parquet")
body = sentences[~sentences["is_boilerplate"]]
print("non-boilerplate sentences:", len(body))
print("target sentences:", int(body["mentions_target"].sum()))

2026-08-16 22:03:31.234 | INFO     | stock_predictor.config:<module>:12 - PROJ_ROOT path is: D:\ML\stock-predictor


non-boilerplate sentences: 57250
target sentences: 13669


## 1. What is being audited

Three mechanisms tag a sentence without the word "Tesla" appearing in it:

| mechanism | how it decides |
|---|---|
| **coreference** | a neural model reads the whole article and links mention chains |
| **anaphora heuristic** | "the company" means whichever company was named most recently, within 6 sentences |
| **CEO alias** | the sentence names Musk; flagged as CEO-related but deliberately NOT tagged as Tesla |

They are mutually exclusive and ordered: an explicit name wins, then coref, then the heuristic as a
fallback. So the heuristic only ever runs on sentences coref could not handle.

## 2. Attempt one, and why it was worthless

The first audit showed an auditor each sentence **on its own** and asked whether it was about Tesla.
It returned coref 0.59, anaphora 0.21, and I nearly reported those as precision.

They are not precision. **Coref and the heuristic resolve pronouns using article context, and I had
removed the context.** For exactly the sentences these mechanisms exist to handle -- "It's a wonderful
business", "the company also saw its first-ever decline in annual revenue" -- the auditor could not
possibly verify the referent, and my instructions told it to answer `no` when no company could be
identified. So "the algorithm was wrong" and "this sheet cannot tell" were collapsed into the same
number. Roughly half the failures were the second kind.

The auditor flagged this itself, unprompted:

> *"the reported `no` rate overstates true error rate and should be read as 'unverifiable from
> sentence alone,' not 'algorithm was wrong.'"*

**The lesson generalises beyond this notebook:** an evaluation must give the judge at least the
evidence the system had. Ours had a whole article; the judge had one sentence.

## 3. Attempt two: context, referent-first, and an explicit ambiguous option

Three changes:

1. **Context.** Each row carries the 5 preceding sentences and 1 following -- matching
   `ANAPHORA_MAX_GAP = 6`, the window the heuristic itself uses.
2. **Ask for the referent, do not ask for a grade.** The first sheet asked "is our claim correct?",
   which invites agreement. The second asks *"what does this phrase refer to?"* as free text and never
   shows our answer. We compare afterwards.
3. **`AMBIGUOUS` is a first-class verdict.** If a careful reader *with* context still cannot tell,
   the confident tag is unjustified either way -- that is a finding about the mechanism, not a hole in
   the audit. Collapsing it into "wrong" is what destroyed attempt one.

To keep (2) honest the substitution preview had the injected company masked to `<NAME>`, so judging
the rewrite could not leak the referent.

### 3.1 The result, and my second mistake

Attempt two gave **coref 74%, anaphora 13%**, with the auditor finding the passage determinate in
**99%** of rows -- so unlike attempt one these were real error rates, not artifacts.

But the substitution numbers were wrong, and again it was my bug. My mask replaced `"Tesla's"` with
`"<NAME>"`, **eating the possessive**. A perfectly correct substitution --

    "Tesla's revenue growth of 25.52% is notably higher..."

was shown to the auditor as

    "<NAME> revenue growth of 25.52% is notably higher..."

which reads as a dropped possessive, and was duly marked broken. 40 of the 85 reported substitution
failures were this artifact. The tell was in the auditor's own report: *"There is not a single
`<NAME>'s` anywhere in the sheet"* -- which should have been impossible if the pipeline were producing
possessives at all, and it was.

Corrected, substitution accuracy was 0.789 rather than 0.60. The referent findings were untouched,
since the mask only ever altered the substitution column.

## 4. What we changed

Three changes, each tied to a measurement.

**Anaphora heuristic switched off** (`USE_ANAPHORA_FALLBACK = False`). 13% correct against coref's
74%, on a mechanism that only fires where coref already failed. The failures are exactly what a
recency rule produces -- 61 of them resolved to a *different company named in the same passage*:

> "**Pilot** is the largest network of travel centers in North America... **The company** serves an
> average of 1.2 million guests per day." → tagged Tesla

> "Ellison owns 41% of **Oracle**... Ellison owned 22% of **the company** 15 years ago" → tagged Tesla

Kept behind a flag rather than deleted, so it can be restored if coref coverage ever regresses.

**First-person plural removed from substitutable anaphora.** `we`/`our`/`us` in news text are nearly
always inside a quote from a *person*: *"'We want the future to look like the future,' Musk said"*
became *"Tesla want the future to look like the future"*.

**Contractions expanded rather than split.** The span covers `It` while the text reads `It's`, so
replacing the span orphaned the clitic: `"It's also profitable"` → `"Tesla's also profitable"` (which
means something different), and `"we're making big investments"` → `"Tesla're making..."`. A clitic
glued to a bare pronoun is now consumed and expanded -- `It's` → `Tesla is`.

In [2]:
# Regenerating with these changes moved the corpus as follows.
deltas = pd.DataFrame({
    "before": [14746, 23213, 4030, 1333, 2465],
    "after": [14433, 22193, 4030, 0, 1925],
}, index=["mentions_target", "mentions_other", "resolved_by_coref",
          "resolved_by_anaphora", "substitutions"])
deltas["delta"] = deltas["after"] - deltas["before"]
deltas

,before,after,delta
mentions_target,14746,14433,-313
mentions_other,23213,22193,-1020
resolved_by_coref,4030,4030,0
resolved_by_anaphora,1333,0,-1333
substitutions,2465,1925,-540


`resolved_by_coref` is **unchanged at 4,030**, which is the check that matters: switching off the
fallback did not disturb the mechanism we kept. `mentions_other` falls by more than `mentions_target`
because the heuristic resolved rival companies too.

## 5. Attempt three: the clean audit

200 coref-resolved sentences from the fixed pipeline, context window intact, referent asked first,
and the mask now mapping `"Tesla's"` → `"<NAME>'s"` so grammar survives and only identity is hidden.
The sheet was asserted to contain `<NAME>'s` rows before being sent -- the previous run's zero was the
bug.

In [3]:
key = pd.read_parquet(PROJ_ROOT / "references" / "context-audit-key-v2.parquet") \
    if (PROJ_ROOT / "references" / "context-audit-key-v2.parquet").exists() else None
lab = pd.read_csv(PROJ_ROOT / "references" / "context-audit-labels-v2.csv")
sheet = pd.read_csv(PROJ_ROOT / "references" / "context-audit-sheet-v2.csv")
d = sheet[["sample_id", "highlighted_phrase"]].merge(lab, on="sample_id")
# pandas reads TRUE/FALSE as booleans and NA as NaN, so normalise via str.
d["ref"] = d["refers_to_tesla"].astype(str).str.upper()
d["span"] = d["substitution_span_ok"].astype(str).str.upper()
d["has_decision"] = d["highlighted_phrase"].notna() & (d["highlighted_phrase"].astype(str) != "")

print(f"rows audited: {len(d)}")
print(f"  carrying a resolution decision: {int(d['has_decision'].sum())}")
print(f"  no phrase resolved            : {int((~d['has_decision']).sum())}")

rows audited: 200
  carrying a resolution decision: 138
  no phrase resolved            : 62


**A sampling flaw the auditor caught.** 62 of the 200 rows have no highlighted phrase at all -- they
are coref-*tagged* but nothing substitutable was found, so there is no resolution decision to judge.
I sampled on the tag rather than on the presence of a span. Those rows are excluded from the rates
below; they are not model errors, and counting them as such would repeat exactly the mistake of
attempt one.

In [4]:
real = d[d["has_decision"]]
print("=== REFERENT (rows carrying a decision) ===")
print(real["ref"].value_counts().to_string())
print(f"precision: {(real['ref'] == 'TRUE').mean():.3f}")
print()
sub = real[real["span"].isin(["TRUE", "FALSE"])]
print(f"=== SUBSTITUTION: {(sub['span'] == 'TRUE').mean():.3f} sound  (n={len(sub)}) ===")
print()
bad = real[(real["ref"] != "TRUE") | (real["span"] == "FALSE")]
print("failure classes:")
print(bad["note"].astype(str).str.lower().value_counts().to_string())

=== REFERENT (rows carrying a decision) ===
ref
TRUE         129
FALSE          8
AMBIGUOUS      1
precision: 0.935

=== SUBSTITUTION: 0.957 sound  (n=138) ===

failure classes:
note
refers to a different company                3
refers to a non-company antecedent           3
plural referent - refers to two companies    2
subject-verb disagreement                    1
antecedent outside context window            1


**Coreference: 93.5% correct referent, 95.7% mechanically sound substitutions.** Only 8 referent
errors in 138 decisions, and 5 of the 6 substitution failures are downstream of a referent error --
just one broke while the referent was right.

The residual errors are a short list, and none has a cheap fix:

- **3 resolved to a different company** (Figure AI, Tractor Supply, SpaceX)
- **3 resolved to a non-company antecedent** -- a percentage, an ETF, a trading alert. *"Tesla's US
  market share dropped to 45%. In 2019, **it** was 80%"* → the antecedent is the figure, and the
  substitution reads *"In 2019, Tesla was 80%."*
- **2 plural referents** collapsed to one company -- *"The problem for **both those companies** was
  **their** expanding capex"* became *"...was Tesla's expanding capex"*
- **1 antecedent above the context window**, the only genuine `AMBIGUOUS`

The auditor found the passage determinate in **137 of 138** rows, so context depth is essentially
never the binding constraint. Whatever headroom remains is in the resolver.

In [5]:
cor = body[body["mentions_target"] & body["resolved_by_coref"]]
print("coref-tagged target sentences:", len(cor))
print(f"  with a usable span (substituted)  : {int(cor['anaphor_char_start'].notna().sum())} "
      f"({100*cor['anaphor_char_start'].notna().mean():.1f}%)")
print(f"  no span (scored unsubstituted)    : {int(cor['anaphor_char_start'].isna().sum())} "
      f"({100*cor['anaphor_char_start'].isna().mean():.1f}%)")

coref-tagged target sentences: 2617
  with a usable span (substituted)  : 1925 (73.6%)
  no span (scored unsubstituted)    : 692 (26.4%)


**The honest caveat on the headline.** 26.4% of coref-tagged sentences carry no usable span, so the
93.5% describes the three quarters that do. The remaining quarter are still tagged as Tesla sentences
and still scored -- on unchanged text, with no aspect anchor -- and this audit says nothing about
whether those tags are right. Some of that quarter exists *because* of the fixes above: a sentence
whose only coref mention was "we" now correctly yields no span, which suppresses the bad substitution
while leaving the tag in place.

That is the next thing to measure, and it is a different question from the one asked here.

## 6. Where this leaves the pipeline

| mechanism | verdict | evidence |
|---|---|---|
| explicit name | keep | 11,052 sentences, a lookup rather than a judgement |
| **coreference (span rows)** | **keep** | 93.5% referent, 95.7% substitution, 2,617 sentences |
| **coreference (no-span rows)** | **keep, flagged noisy** | 56.5% referent (95% CI [49.0%, 63.7%], n=170) -- see notebook 2.7 |
| **coreference (blended, population-weighted)** | -- | **78.3%**, not 93.5% -- see notebook 2.7 |
| **anaphora heuristic** | **off** | 13% referent; cost 313 target sentences (2.1%) |
| CEO alias | unchanged | never sets `mentions_target`; unaudited here |

Final corpus: **13,669 non-boilerplate target sentences** — 11,052 explicit, 2,617 coref, 0 heuristic.

### Limitations

- **The 93.5% below is a SPAN-ONLY figure, not a channel figure.** It describes the rows carrying a
  decision at the time this notebook was written. Notebook 2.7 closes that gap: the no-span quarter
  (originally 26.4% of coref tags, measured at 1,189 rows / 23.5% of the current corpus) runs at
  56.5% correct (n=170), and the population-weighted blend across both populations is **78.3%**.
  Any statement of "coref accuracy" outside this notebook should cite the 78.3% blended figure, not
  93.5%.
- **The auditor is an LLM**, so this is protocol-driven adjudication, not human validation. It is
  reproducible and it was blind to our answers, which is the most that is available cheaply.
- **"The stock" was counted as Tesla.** 13-15 rows resolve a phrase to Tesla's *shares* rather than
  the company. If a downstream feature ever needs those separated, the precision figure drops to
  roughly 83%.
- **Three rows needed world knowledge** rather than an in-window antecedent (a Q3 delivery report
  among mega-caps; "the EV maker... its 10 millionth vehicle"). A resolver without that knowledge
  would legitimately fail them.
- **Nothing here measures recall.** We audited what the pipeline claimed, never what it missed. A
  sentence that should have been tagged and was not is invisible to this design.

### On the method itself

Two of the three attempts were invalidated by my own design errors -- removing the context the system
had, then masking away the grammar the auditor was asked to judge. Both were caught by the auditor's
own report rather than by me, and in both cases the tell was a number that was *too extreme to be
plausible* (67% of everything wrong; not one possessive in 213 substitutions). That is the practical
heuristic worth keeping: when an evaluation says almost everything is broken, suspect the evaluation
first.